# 🏭 Hosting de ComfyUI en Colab — Datos sintéticos de defectos en rollos de aluminio

Este notebook levanta **ComfyUI** en una GPU T4 gratis, descarga los modelos base (SDXL RealVisXL + VAE + Fooocus inpaint) y te da una **URL pública** para usar la interfaz.

**Orden de uso:**
1. `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.
2. Ejecuta las celdas 1→7 en orden.
3. Abre la URL `trycloudflare.com` que imprime la última celda.
4. En ComfyUI, arrastra `workflows/inpaint_defectos.json` a la ventana.

> Las celdas son **idempotentes**: puedes re-ejecutarlas sin duplicar descargas.

## 1 · Verificar GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA disponible:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (activa la GPU en el menu)')

## 2 · Configuración y (opcional) montar Google Drive
Edita las variables si hace falta. Drive sirve para traer tu LoRA y guardar salidas.

In [ ]:
# === Configuracion ===
USAR_DRIVE   = True   # monta Google Drive (LoRA de entrada / salidas persistentes)
DRIVE_BASE   = '/content/drive/MyDrive/rollos_dataset'
CIVITAI_TOKEN = ''    # opcional: token de Civitai para descargas programaticas
HF_TOKEN      = ''    # opcional: token de Hugging Face (modelos con login)

import os
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_BASE, exist_ok=True)
    print('Drive montado en', DRIVE_BASE)

## 3 · Instalar ComfyUI + ComfyUI-Manager (+ nodos de inpaint Fooocus)

In [ ]:
%cd /content
import os
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# ComfyUI-Manager (instalar/gestionar custom nodes desde la UI)
if not os.path.exists('custom_nodes/ComfyUI-Manager'):
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager custom_nodes/ComfyUI-Manager

# (opcional) nodos de inpaint Fooocus -> mejor calidad de inpainting SDXL
if not os.path.exists('custom_nodes/comfyui-inpaint-nodes'):
    !git clone https://github.com/Acly/comfyui-inpaint-nodes custom_nodes/comfyui-inpaint-nodes
print('ComfyUI instalado.')

## 4 · Descargar modelos base (aria2c multihilo)
SDXL RealVisXL V5.0 + VAE fp16-fix + Fooocus inpaint (head + patch).

In [ ]:
!apt-get -qq install -y aria2 >/dev/null
import os
CK  = '/content/ComfyUI/models/checkpoints'
VAE = '/content/ComfyUI/models/vae'
INP = '/content/ComfyUI/models/inpaint'
for d in (CK, VAE, INP):
    os.makedirs(d, exist_ok=True)

def dl(url, dest_dir, out):
    if os.path.exists(os.path.join(dest_dir, out)):
        print('ya existe:', out); return
    !aria2c -x16 -s16 -k1M --console-log-level=warn -d "{dest_dir}" -o "{out}" "{url}"

# Checkpoint base SDXL fotorrealista
dl('https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0_fp16.safetensors',
   CK, 'RealVisXL_V5.0_fp16.safetensors')
# VAE fp16-fix (evita artefactos/NaN en T4)
dl('https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors',
   VAE, 'sdxl_vae.safetensors')
# Fooocus inpaint (upgrade de calidad, opcional en el workflow)
dl('https://huggingface.co/lllyasviel/fooocus_inpaint/resolve/main/fooocus_inpaint_head.pth',
   INP, 'fooocus_inpaint_head.pth')
dl('https://huggingface.co/lllyasviel/fooocus_inpaint/resolve/main/inpaint_v26.fooocus.patch',
   INP, 'inpaint_v26.fooocus.patch')
print('Modelos base listos.')

## 5 · Copiar tu LoRA de defectos desde Drive
Tras la **FASE B** (entrenamiento) tendrás `rollo_defectos_lora.safetensors` en Drive.
Esta celda lo copia a `models/loras`. Si aún no existe, no pasa nada.

In [ ]:
import os, shutil, glob
LORA_DIR = '/content/ComfyUI/models/loras'
os.makedirs(LORA_DIR, exist_ok=True)
copiados = 0
if USAR_DRIVE and os.path.isdir(DRIVE_BASE):
    for f in glob.glob(os.path.join(DRIVE_BASE, '**', '*.safetensors'), recursive=True):
        nombre = os.path.basename(f).lower()
        if 'lora' in nombre or 'defect' in nombre:
            shutil.copy(f, LORA_DIR); copiados += 1
            print('copiado LoRA:', os.path.basename(f))
if copiados == 0:
    print('Sin LoRA todavia: entrenalo en train_LoRA_kohya.ipynb (FASE B) o subelo a', LORA_DIR)

## 6 · (Opcional) Descargar un modelo extra de Civitai / Hugging Face
Pega una URL de descarga directa. Soporta token de Civitai y de HF.

In [ ]:
URL_EXTRA = ''     # p.ej. una URL de Civitai o HF resolve/main/...
NOMBRE    = ''     # nombre de archivo de salida, p.ej. 'JuggernautXL.safetensors'
DEST      = CK     # CK=checkpoints | VAE | INP=inpaint | '/content/ComfyUI/models/loras'

if URL_EXTRA and NOMBRE:
    url = URL_EXTRA; hdr = ''
    if 'civitai.com' in url and CIVITAI_TOKEN:
        url += ('&' if '?' in url else '?') + 'token=' + CIVITAI_TOKEN
    if 'huggingface.co' in url and HF_TOKEN:
        hdr = '--header="Authorization: Bearer ' + HF_TOKEN + '"'
    !aria2c -x16 -s16 {hdr} --console-log-level=warn -d "{DEST}" -o "{NOMBRE}" "{url}"
else:
    print('Deja URL_EXTRA y NOMBRE vacios si no necesitas modelos extra.')

## 7 · Lanzar ComfyUI + túnel público (cloudflared)
**Deja esta celda ejecutándose.** Imprime una URL `https://...trycloudflare.com` — ábrela.
Al terminar, arrastra un workflow de `workflows/` a la ventana de ComfyUI.

In [ ]:
import subprocess, time, os, urllib.request

# instalar cloudflared si falta
if not (os.path.exists('/usr/local/bin/cloudflared') or os.path.exists('/usr/bin/cloudflared')):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

# arrancar ComfyUI en segundo plano (--lowvram para caber en T4)
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188',
     '--lowvram', '--preview-method', 'auto'],
    cwd='/content/ComfyUI')

print('Esperando a que ComfyUI arranque...')
ok = False
for _ in range(90):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188', timeout=2); ok = True; break
    except Exception:
        time.sleep(3)
print('ComfyUI arriba.' if ok else 'ComfyUI tardo; revisa el log si el tunel no responde.')

print('\nAbriendo tunel cloudflared. Busca la URL *.trycloudflare.com abajo:\n')
!cloudflared tunnel --url http://127.0.0.1:8188